# NGL BlobMorph notebook
Creates a morph matrix (morph levels x stimulus sizes) between two blob objects, following Rhee et al. (2025, *Cell Reports*, Fig. 1A and Fig. S4A) and Zoccolan et al. (2009, *PNAS*). Same parameters and results as `matlab/blobMorph_master.m` and `matlab/blobMorph_live.m`.

1. Section 0: image bank. Set `DOWNLOAD = True` once to download the images.
2. Sections 1 to 10: parameters. Section 1 is required; the other sections have working defaults.
3. Section 11: checks of the settings.
4. Section 12: morph.
5. Sections 13 to 17: results and location of the files.

Run all cells: *Run > Run All Cells*. Jupyter is started in the `NGL-BlobMorph` folder or in one of its subfolders (README, Jupyter notebook).

Author: Jesus J. Ballesteros. NGL BlobMorph, MIT Licence (see `LICENSE`).

In [ ]:
import os
import sys

# NGL-BlobMorph folder: first folder holding blobmorph/, from the working folder upwards
ROOT = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(ROOT, "blobmorph", "__init__.py")):
    if os.path.dirname(ROOT) == ROOT:
        raise RuntimeError("NGL-BlobMorph folder not found: start Jupyter in the NGL-BlobMorph folder.")
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import csv

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import PIL
import scipy
from IPython.display import Image as ShowImage, display
from PIL import Image

import blobmorph
from blobmorph import pipeline
from blobmorph.download import download

print("NGL BlobMorph %s: %s" % (blobmorph.__version__, ROOT))

## 0. Image bank and folders
Image bank: folder `images` of https://github.com/coxlab/povray_blobs (Cox lab; not distributed with NGL BlobMorph). `DOWNLOAD = True` saves the images of `IMAGE_SET` in `Images/<IMAGE_SET>` of the NGL-BlobMorph folder. Results are written to `results/<RESULTS_FOLDER>`.

In [ ]:
IMAGE_SET = "Blobs_TrainingRatsD1D2"  # folder of images/ in coxlab/povray_blobs
DOWNLOAD = False                     # True: download the image set (once)
RESULTS_FOLDER = "morph_N1_N2"       # results/<RESULTS_FOLDER>

DATA_DIR = os.path.join(ROOT, "Images", IMAGE_SET)
OUTPUT_DIR = os.path.join(ROOT, "results", RESULTS_FOLDER)
if DOWNLOAD:
    download(IMAGE_SET)
if os.path.isdir(DATA_DIR):
    n_png = len([f for f in os.listdir(DATA_DIR) if f.lower().endswith(".png")])
    print("Image bank: %s (%d images)" % (DATA_DIR, n_png))
else:
    print("Image bank not found: %s\nSet DOWNLOAD = True and run this cell again." % DATA_DIR)
print("Results: %s" % OUTPUT_DIR)

## 1. Input images (required)
`OBJECT_A`: morph level 0 %. `OBJECT_B`: morph level 100 %. Image bank: `"Blob_N1"` (object 1) or `"Blob_N2"` (object 2). `ROTATION`: view of both objects, rotation about the vertical axis, -90 to 90 deg in steps of 15 deg; 0 is the default view of Zoccolan et al. (2009), Fig. 1A. `OTHER_IMAGE_A`, `OTHER_IMAGE_B`: other image files, used instead of the selection: grayscale renders on a black background, both of the same size. With `METHOD = "models"` (section 5) no images are needed.

In [ ]:
OBJECT_A = "Blob_N1"  # "Blob_N1" or "Blob_N2"
OBJECT_B = "Blob_N2"
ROTATION = 0          # -90 to 90 in steps of 15 (deg)
OTHER_IMAGE_A = ""    # e.g. r"C:\data\object_A.png"; "": not used
OTHER_IMAGE_B = ""

IMAGE_A = OTHER_IMAGE_A or os.path.join(DATA_DIR, "%s_CamRot_y%d.png" % (OBJECT_A, ROTATION))
IMAGE_B = OTHER_IMAGE_B or os.path.join(DATA_DIR, "%s_CamRot_y%d.png" % (OBJECT_B, ROTATION))
print("ImageA: %s\nImageB: %s" % (IMAGE_A, IMAGE_B))

## 2. Morph matrix
`NUM_LEVELS`: morph levels, both objects included; 9 in Fig. S4A, 22 in the behavioural morph set. `SIZES`: degrees of visual angle; `[10, 20, 30, 40, 50]` in Fig. S4A, `[15, 20, 25, 30, 35, 40]` in Zoccolan et al. (2009), Fig. 2. `SAMPLING`, spacing of the levels:
- `"uniform"`: equal steps of the morph parameter (levels of Fig. S4A)
- `"arc"`: equal pixel-space Euclidean distance along the morph (Rhee et al. 2025, Methods)
- `"chord"`: equal Euclidean distance between successive chosen levels
- `"original"`: procedure of julianarhee/morph-pov, 2000 morphs, about 25 min

In [ ]:
NUM_LEVELS = 9
SIZES = [10, 20, 30, 40, 50]
SAMPLING = "uniform"  # "uniform", "arc", "chord" or "original"

## 3. Display: degrees to pixels
`PIXELS_PER_DEGREE`: pixels per degree of visual angle; 16.05 for the two-photon imaging display of Rhee et al. (2025). `None`: screen geometry instead, `SCREEN_WIDTH_CM` and `VIEWING_DISTANCE_CM` (Zoccolan et al. 2009: Samsung SyncMaster 940BX, 1280 x 1024 px, 37.6 cm wide, eyes at 25 cm). `SCREEN_PIXELS`: [width, height] of the screen, also the size of the full-screen images. `DEGREE_MODE` (screen geometry): `"mworks"`, linear in degrees over the whole screen; `"tangent"`, exact visual angle at the screen centre; `"linear"`, small-angle approximation. `STIMULUS_POSITION_DEG`: centre on the screen images, [x, y] degrees from the screen centre. `SIZE_REFERENCE`: dimension that spans the size: `"crop_max"` (longest side of the common crop of all levels), `"crop_width"`, `"crop_height"` or `"object_max"` (each object's own bounding box).

In [ ]:
PIXELS_PER_DEGREE = 16.05
SCREEN_PIXELS = [1920, 1080]
SCREEN_WIDTH_CM = None
VIEWING_DISTANCE_CM = None
DEGREE_MODE = "mworks"
STIMULUS_POSITION_DEG = [0, 0]
SIZE_REFERENCE = "crop_max"

## 4. Luminance controls and figure
`DISPLAY_GAMMA`: gamma of the display for the luminance-matched full-field grey levels (1 for a linearised display). `SAVE_SCREENS`: full-screen image of each stimulus. `LUMINANCE_DISPLAY`: luminance column of the figure, `"actual"` (computed full-field grey level of each size) or `"schematic"` (255 x size / largest size, as drawn in Fig. S4A). `FIGURE_TITLE`: text above the figure (`""`: none).

In [ ]:
DISPLAY_GAMMA = 2.2
SAVE_SCREENS = True
LUMINANCE_DISPLAY = "schematic"
FIGURE_TITLE = ""

## 5. Object models
`METHOD`:
- `"auto"`: recognise the two reference objects (Zoccolan et al. 2009), otherwise fit models to the images
- `"preset"`: reference objects only
- `"fit"`: blob models fitted to the images (about 11 min per image)
- `"models"`: `MODEL_A` and `MODEL_B` (POV-Ray `.pov` scene or `.json` model)

Model examples: `tests/povray_reference/obj1_cam11_v36.pov` and `obj2_cam11_v36.pov` (scene files of the reference objects); `examples/fitted_from_images/models/fitted_A.json` and `fitted_B.json` (models fitted in a previous run). `POV_DECLARE`: values of identifiers left free in `.pov` files, e.g. `{"obj_pos_z": 6}`. `SETUP`, rendering set-up for fitted objects and model files: `"auto"` (fitted objects: camera z = -11, tone curve zoccolan, antialiasing as the input images; fitted models `.json`: set-up stored in the file; `.pov` files: camera of the scene file, tone curve calibrated on the images, linear without images), `"provided"` (camera z = -11, tone curve zoccolan), `"zoccolan2009"` (camera z = -10, linear output), `"rhee2025"` (camera z = -10, sRGB output, no antialiasing).

In [ ]:
METHOD = "auto"  # "auto", "preset", "fit" or "models"
MODEL_A = ""     # e.g. os.path.join(ROOT, "examples", "fitted_from_images", "models", "fitted_A.json")
MODEL_B = ""
POV_DECLARE = {}
SETUP = "auto"   # "auto", "provided", "zoccolan2009" or "rhee2025"

## 6. Correspondence between object parts
Objects other than the reference pair. `CORRESPONDENCE`: `"match"`, matched parts are interpolated and the others fade; `"crossfade"`, all parts of A fade out while all parts of B fade in; `"index"`, part i of A becomes part i of B. `VANISH`, parts without partner: `"fade"` (strength to 0) or `"shrink"` (radius to 0). `INTERPOLATION`: `"pov"`, linear on the POV-Ray numbers; `"geometric"`, centre, semi-axes and orientation of each ellipsoid; `"auto"`, geometric for fitted models.

In [ ]:
CORRESPONDENCE = "match"
VANISH = "fade"
INTERPOLATION = "auto"

## 7. Image fitting
Used with `METHOD = "fit"`, or `"auto"` when the objects are not recognised. `stages`: one row per stage, [resolution factor, supersampling, maximum generations, initial step].

In [ ]:
FIT = {
    "symmetric": "auto",   # left-right symmetric model: "auto", True or False
    "max_components": 4,   # lobes taken from the silhouette (maximum)
    "max_extra": 2,        # parts added inside the silhouette (maximum)
    "restarts": 1,         # runs of the first stage
    "popsize": 16,         # CMA-ES population
    "seed": 0,             # random seed
    "stages": [[0.15, 2, 160, 1.0],
               [0.30, 2, 70, 0.3],
               [0.50, 1, 40, 0.1]],
}

## 8. Post-processing of the rendered frames
`USE_INPUTS_AS_ANCHORS`: levels 0 % and 100 % are the input images instead of renders. `CENTER`: `"anchors"`, joint bounding box of both objects centred in the frame, or `"none"`; `CENTER_HORIZONTAL`: also horizontally. `CROP_MARGIN`: pixels around the joint bounding box of all levels. `ANTIALIAS`: `"auto"` (as the input images), `True` or `False`. `SAVE_FRAMES`: full-size frames. `SAVE_POV`: POV-Ray scene of each level. `LUMINANCE_COLUMN`: luminance column in the figure.

In [ ]:
USE_INPUTS_AS_ANCHORS = False
CENTER = "anchors"
CENTER_HORIZONTAL = True
CROP_MARGIN = 2
ANTIALIAS = "auto"
SAVE_FRAMES = True
SAVE_POV = True
LUMINANCE_COLUMN = True

## 9. Performance
`NUM_DENSE`: morphs rendered to measure pixel distances (`None`: 201; 2002 for `"original"`). `DENSE_SCALE`: resolution factor of these renders. `DENSE_SUPERSAMPLE`: rays per pixel side. `PROCESSES`: parallel processes (`None`: min(8, number of cores / 2)).

In [ ]:
NUM_DENSE = None
DENSE_SCALE = 0.5
DENSE_SUPERSAMPLE = 2
PROCESSES = None

## 10. Environment
Python of the notebook kernel. `VERBOSE`: progress messages.

In [ ]:
VERBOSE = True
print("Python %s | numpy %s | scipy %s | pillow %s | matplotlib %s" % (
    sys.version.split()[0], np.__version__, scipy.__version__, PIL.__version__,
    matplotlib.__version__))

## 11. Checks
Lists the settings and stops with a message when an input is missing or a value is out of range.

In [ ]:
problems = []
use_images = METHOD != "models" or (os.path.isfile(IMAGE_A) and os.path.isfile(IMAGE_B))
if METHOD == "models":
    for name, path in (("MODEL_A", MODEL_A), ("MODEL_B", MODEL_B)):
        if not os.path.isfile(path):
            problems.append("%s not found: %s" % (name, path))
else:
    for name, path in (("ImageA", IMAGE_A), ("ImageB", IMAGE_B)):
        if not os.path.isfile(path):
            problems.append("%s not found: %s (image bank: section 0, DOWNLOAD = True)" % (name, path))
if use_images and not problems and Image.open(IMAGE_A).size != Image.open(IMAGE_B).size:
    problems.append("ImageA and ImageB differ in size")
if int(NUM_LEVELS) < 2:
    problems.append("NUM_LEVELS: 2 or more")
if not SIZES or min(SIZES) <= 0:
    problems.append("SIZES: positive values")
if PIXELS_PER_DEGREE is None and not (SCREEN_WIDTH_CM and VIEWING_DISTANCE_CM):
    problems.append("give PIXELS_PER_DEGREE, or SCREEN_WIDTH_CM and VIEWING_DISTANCE_CM")

print("Images:      %s" % ("%s\n             %s" % (IMAGE_A, IMAGE_B) if use_images else "none"))
print("Levels:      %d (%s)" % (NUM_LEVELS, SAMPLING))
print("Sizes (deg): %s" % ", ".join("%g" % s for s in SIZES))
print("Method:      %s" % METHOD)
print("Results:     %s" % OUTPUT_DIR)
if problems:
    raise ValueError("\n".join(problems))
print("Settings OK.")

## 12. Morph
Creates the morph matrix. Duration on 8 cores: 15 to 30 s (reference objects, uniform); about 2 min (arc or chord); about 25 min (original, or objects fitted to the images). `config` holds the settings with the option names of the Python back end, the same names as in a settings file for `python -m blobmorph --config FILE`.

In [ ]:
config = {
    "image_a": IMAGE_A if use_images else None,
    "image_b": IMAGE_B if use_images else None,
    "model_a": MODEL_A or None,
    "model_b": MODEL_B or None,
    "pov_declare": POV_DECLARE,
    "method": METHOD,
    "setup": SETUP,
    "n_levels": int(NUM_LEVELS),
    "sizes": [float(s) for s in SIZES],
    "sampling": SAMPLING,
    "n_dense": NUM_DENSE,
    "dense_scale": DENSE_SCALE,
    "dense_supersample": DENSE_SUPERSAMPLE,
    "correspondence": CORRESPONDENCE,
    "vanish": VANISH,
    "interpolation": INTERPOLATION,
    "antialias": ANTIALIAS,
    "center": CENTER,
    "center_horizontal": CENTER_HORIZONTAL,
    "crop_margin": CROP_MARGIN,
    "size_reference": SIZE_REFERENCE,
    "px_per_deg": PIXELS_PER_DEGREE,
    "screen_px": list(SCREEN_PIXELS),
    "screen_cm": SCREEN_WIDTH_CM,
    "distance_cm": VIEWING_DISTANCE_CM,
    "deg_mode": DEGREE_MODE,
    "stim_pos_deg": list(STIMULUS_POSITION_DEG),
    "save_screens": SAVE_SCREENS,
    "display_gamma": DISPLAY_GAMMA,
    "luminance_column": LUMINANCE_COLUMN,
    "luminance_display": LUMINANCE_DISPLAY,
    "use_inputs_as_anchors": USE_INPUTS_AS_ANCHORS,
    "save_frames": SAVE_FRAMES,
    "save_pov": SAVE_POV,
    "processes": PROCESSES,
    "fit": FIT,
    "title": FIGURE_TITLE or None,
    "out_dir": OUTPUT_DIR,
    "verbose": VERBOSE,
}
manifest = pipeline.run(config)

## 13. Results: morph matrix
File: `figures/morph_matrix.png` (also `morph_matrix.pdf`; without labels: `morph_matrix_panel.png`) in the results folder.

In [ ]:
figure_file = os.path.join(OUTPUT_DIR, "figures", "morph_matrix.png")
print(figure_file)
display(ShowImage(filename=figure_file))

## 14. Results: all morph levels at the largest size
Files: `stimuli/morphLL_levelPPP.PP_sizeSSS.S.png` (LL: level index, PPP.PP: morph level in %, SSS.S: size in degrees). Full-screen versions: `screens/`.

In [ ]:
levels = manifest["levels_percent"]
sizes = manifest["sizes_deg"]
files = manifest["stimulus_files"]  # [size][level], relative to OUTPUT_DIR
k = int(np.argmax(sizes))
fig, axes = plt.subplots(1, len(levels), figsize=(1.8 * len(levels), 2.2))
for j, ax in enumerate(np.atleast_1d(axes)):
    ax.imshow(Image.open(os.path.join(OUTPUT_DIR, files[k][j])), cmap="gray", vmin=0, vmax=255)
    ax.set_title("%g %%" % levels[j], fontsize=9)
    ax.axis("off")
fig.suptitle("size %g deg" % sizes[k])
plt.show()

## 15. Results: middle morph level at all sizes
Stimuli at their size in pixels.

In [ ]:
j = len(levels) // 2
stims = [np.asarray(Image.open(os.path.join(OUTPUT_DIR, files[i][j]))) for i in range(len(sizes))]
gap = 20
height = max(s.shape[0] for s in stims)
width = sum(s.shape[1] for s in stims) + gap * (len(stims) - 1)
canvas = np.zeros((height, width), np.uint8)
x = 0
for s in stims:
    y = (height - s.shape[0]) // 2
    canvas[y:y + s.shape[0], x:x + s.shape[1]] = s
    x += s.shape[1] + gap
plt.figure(figsize=(12, 12 * height / width + 0.5))
plt.imshow(canvas, cmap="gray", vmin=0, vmax=255)
plt.title("morph level %g %%, sizes %s deg" % (levels[j], ", ".join("%g" % s for s in sizes)))
plt.axis("off")
plt.show()

## 16. Results: pixel distance between successive morph levels
Euclidean distance between the cropped full-size frames of successive levels. File: `figures/distance_profile.png`, which also shows the cumulative distance along the morph.

In [ ]:
display(ShowImage(filename=os.path.join(OUTPUT_DIR, "figures", "distance_profile.png")))

## 17. Results: list of stimuli and location of the files
`manifest.csv`: one row per stimulus (morph level, morph parameter t, size, file names, size in pixels, luminance-matched full-field grey level). `manifest.json`: all settings and results.

In [ ]:
with open(os.path.join(OUTPUT_DIR, "manifest.csv"), newline="") as f:
    rows = list(csv.DictReader(f))
columns = ["morph_level_percent", "size_deg", "width_px", "height_px", "luminance_ff_level",
           "stimulus_file"]
print("  ".join("%-10s" % c[:10] for c in columns[:-1]), columns[-1])
for r in rows:
    print("  ".join("%-10s" % r[c] for c in columns[:-1]), r[columns[-1]])

print("\nResults folder: %s" % OUTPUT_DIR)
for sub in ("figures", "stimuli", "screens", "frames", "models"):
    folder = os.path.join(OUTPUT_DIR, sub)
    if os.path.isdir(folder):
        print("  %-9s %3d files" % (sub + "/", len(os.listdir(folder))))
print("  manifest.csv, manifest.json")